# 10 - Financial Feature Engineering

**Project:** AI Quantitative Trading & Strategy Optimisation System  
**Phase:** 6 — Feature Engineering and Market Regime Detection  
**Status:** Executed against the recorded local acquisitions on 14 November 2026; owner review is pending.

## 1. Objective and scope

Define a compact, reproducible set of daily financial features from an instrument’s own closing prices and volume. Calculations are asset-agnostic and do not train a forecasting model, construct a trading strategy, or make an execution assumption.

Feature values indexed by session *t* use observations no later than the close of *t*. They are therefore available only after that close; any later research decision must execute no earlier than the next eligible session. The notebook does not shift the features because it records their information timestamp explicitly.

## 2. Questions and prior evidence

This work follows the documented return, stationarity, autocorrelation and volatility analyses in Notebook 09. Those analyses found evidence of serial dependence in returns and squared returns, but did not establish predictability or approve a particular feature. Phase 3 also documented price adjustment, corporate-action, missing-observation and OHLC limitations. Those earlier audits are not repeated here.

Questions for this notebook:

1. Which simple return, momentum, trend, volatility, drawdown and volume transformations can be calculated reproducibly from the acquired daily series?
2. Do definitions preserve missing observations and use only current and past observations?
3. Which plausible features must be deferred because their inputs or adjustment semantics are not sufficiently validated?

## 3. Candidate definitions

For closing price *Pₜ*, simple return is *Pₜ / Pₜ₋₁ − 1* and log return is *log(Pₜ) − log(Pₜ₋₁)*. Momentum over *w* sessions is *Pₜ / Pₜ₋w − 1*. SMA and EMA use the current close and up to *w − 1* preceding closes; price-to-SMA is *Pₜ / SMAₜ,w − 1*. Return volatility is the sample standard deviation of one-session simple returns over *w* observations. Realised volatility is the square root of the sum of squared log returns over *w* observations. Downside volatility is the root mean square of negative simple returns, with non-negative returns contributing zero. Drawdown is *Pₜ / max(Pₜ₋w+1 … Pₜ) − 1*.

Volume change is its one-session percentage change. Relative volume divides current volume by the mean of the preceding *w* sessions, excluding the current session from the baseline. Candidate windows are 5, 10, 20, 50 and 100 sessions. Rolling features require complete windows; there is no imputation or automatic removal of missing observations.

## 4. Input and unit decisions

The acquired market Parquet files contain `Close`, `Adj Close`, OHLC fields and `Volume` (the FTSE file has no corporate-action columns). This implementation uses the provider’s `Close` field for all price features. It does not convert GBp to GBP. Returns and ratios are scale-invariant, while price-level outputs retain the provider’s original units. Asset configurations identify GBp for the three equities, but historical denomination validation remains pending under D140.

Using `Close` means returns are price returns and do not include dividends. Corporate actions or source revisions can still create discontinuities. `Adj Close` was not selected because its historical adjustment semantics and point-in-time availability have not been independently validated for this research use. Raw files remain unchanged.

High–low range and true range are deferred: mixing adjusted closing prices with unadjusted OHLC can create inconsistent ranges, and a validated adjustment convention is not yet available. Cumulative-from-sample-start returns are also omitted because the arbitrary start point adds little information beyond returns and momentum. No feature is assumed useful merely because it can be calculated.

## 5. Temporal and missing-data controls

Inputs must have a chronological, unique index. Missing values remain missing. Percentage changes use no fill method; rolling statistics require the full specified window. No data are resampled, interpolated, forward-filled, normalised using full-sample statistics, or joined to external series. In particular, current-session close-based features are not represented as available before the session close.

## 6. Implementation and reproducibility

Calculations are implemented in `src/features/financial.py`, separately from immutable Raw data and reusable across configured assets. Unit tests cover formulae, scale invariance, missing data, index ordering, and invalid inputs. The notebook was executed against the recorded local acquisitions on 14 November 2026; outputs are stored in this file. Code execution is evidence of calculation, not proof of data correctness or predictive value.


In [1]:
from pathlib import Path
import sys

import pandas as pd
import yaml

# Resolve the repository root without assuming the notebook's launch directory.
PROJECT_ROOT = next(
    parent for parent in (Path.cwd(), *Path.cwd().parents)
    if (parent / "configs" / "assets").is_dir()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.features.financial import DEFAULT_LOOKBACKS, build_financial_features

ASSET_CONFIG_PATHS = sorted((PROJECT_ROOT / "configs" / "assets").glob("*.yaml"))
assert ASSET_CONFIG_PATHS, "No configured market assets were found."
print(f"Repository: {PROJECT_ROOT}")
print(f"Configured assets: {len(ASSET_CONFIG_PATHS)}")
print(f"Candidate lookbacks: {DEFAULT_LOOKBACKS}")

Repository: /home/anderjcruz/projects/Quantitative-Trading-Strategy
Configured assets: 4
Candidate lookbacks: (5, 10, 20, 50, 100)


## 7. Load the recorded market acquisitions

Each asset is discovered from its YAML configuration. The notebook requires one matching acquired Parquet file for each configured provider symbol; it does not silently pick among multiple versions. Missing or ambiguous Raw acquisitions stop the notebook for review. Loading is read-only.

In [2]:
market_data = {}
source_records = []

for config_path in ASSET_CONFIG_PATHS:
    with config_path.open(encoding="utf-8") as config_file:
        config = yaml.safe_load(config_file)

    symbol = config["provider"]["symbol"]
    asset_name = config["asset"]["name"]
    candidates = sorted(
        (PROJECT_ROOT / "data" / "raw" / "market" / "yahoo_finance" / symbol)
        .glob(f"acquisition_*/{symbol}.parquet")
    )
    if len(candidates) != 1:
        raise RuntimeError(
            f"Expected exactly one recorded Parquet acquisition for {symbol}; "
            f"found {len(candidates)}. Select and document an acquisition before proceeding."
        )

    source_path = candidates[0]
    frame = pd.read_parquet(source_path)
    if "Date" in frame.columns:
        frame = frame.set_index("Date")
    frame.index = pd.to_datetime(frame.index)
    frame.index.name = "observation_date"
    if not frame.index.is_monotonic_increasing or frame.index.has_duplicates:
        raise ValueError(f"Dates for {symbol} must be unique and chronological.")
    if "Close" not in frame.columns:
        raise ValueError(f"The recorded acquisition for {symbol} has no Close column.")

    market_data[symbol] = frame
    source_records.append({
        "symbol": symbol,
        "asset_name": asset_name,
        "source_path": source_path.relative_to(PROJECT_ROOT).as_posix(),
        "rows": len(frame),
        "first_observation": frame.index.min(),
        "last_observation": frame.index.max(),
        "price_field": "Close",
        "original_price_unit": config["asset"]["price_unit"],
        "unit_validation_status": config["asset"]["price_unit_validation_status"],
    })

source_inventory = pd.DataFrame(source_records).set_index("symbol")
source_inventory

,asset_name,source_path,rows,first_observation,last_observation,price_field,original_price_unit,unit_validation_status
symbol,,,,,,,,
^FTSE,FTSE 100 Index,data/raw/market/yahoo_finance/^FTSE/acquisitio...,11004,1984-01-03,2026-11-02,Close,index_points,not_applicable
NG.L,National Grid plc,data/raw/market/yahoo_finance/NG.L/acquisition...,7890,1995-12-11,2026-11-02,Close,GBp,pending
SSE.L,SSE plc,data/raw/market/yahoo_finance/SSE.L/acquisitio...,9060,1991-06-17,2026-11-02,Close,GBp,pending
SVT.L,Severn Trent plc,data/raw/market/yahoo_finance/SVT.L/acquisitio...,9455,1989-12-11,2026-11-02,Close,GBp,pending


## 8. Build the feature tables

The function preserves each asset’s own observation calendar. It does not align assets to one another, fill calendar gaps or convert units. Index and equity features can be compared as dimensionless returns and ratios; price-level SMA/EMA fields remain in their source units.

In [3]:
financial_features = {
    symbol: build_financial_features(frame, price_column="Close", volume_column="Volume")
    for symbol, frame in market_data.items()
}

feature_inventory = pd.DataFrame([
    {
        "symbol": symbol,
        "observations": len(features),
        "feature_columns": len(features.columns),
        "first_feature_date": features.index.min(),
        "last_feature_date": features.index.max(),
        "missing_feature_values": int(features.isna().sum().sum()),
    }
    for symbol, features in financial_features.items()
]).set_index("symbol")
feature_inventory

,observations,feature_columns,first_feature_date,last_feature_date,missing_feature_values
symbol,,,,,
^FTSE,11004,48,1984-01-03,2026-11-02,115491
NG.L,7890,48,1995-12-11,2026-11-02,1781
SSE.L,9060,48,1991-06-17,2026-11-02,7194
SVT.L,9455,48,1989-12-11,2026-11-02,10211


## 9. Descriptive redundancy screen

A full-sample Spearman rank-correlation screen identifies pairs of candidate features with absolute correlation above 0.95 and at least 252 paired observations. This is an exploratory redundancy diagnostic only: overlapping observations, non-stationarity, multiple comparisons and full-sample use make it unsuitable for selecting model features or claiming predictive value.


In [4]:
from itertools import combinations

redundancy_rows = []
for symbol, features in financial_features.items():
    for first, second in combinations(features.columns, 2):
        paired = features[[first, second]].dropna()
        if len(paired) < 252:
            continue
        correlation = paired[first].corr(paired[second], method="spearman")
        if pd.notna(correlation) and abs(correlation) > 0.95:
            redundancy_rows.append({
                "symbol": symbol,
                "feature_a": first,
                "feature_b": second,
                "spearman_rho": correlation,
                "paired_observations": len(paired),
            })

redundancy_screen = pd.DataFrame(redundancy_rows)
redundancy_summary = (
    redundancy_screen.groupby("symbol").agg(
        highly_correlated_pairs=("spearman_rho", "size"),
        median_absolute_rho=("spearman_rho", lambda values: values.abs().median()),
    )
    if not redundancy_screen.empty
    else pd.DataFrame(columns=["highly_correlated_pairs", "median_absolute_rho"])
)
redundancy_summary, redundancy_screen.sort_values("spearman_rho", key=lambda values: values.abs(), ascending=False).groupby("symbol").head(5)

(        highly_correlated_pairs  median_absolute_rho
 symbol                                              
 NG.L                         52             0.995843
 SSE.L                        52             0.995769
 SVT.L                        51             0.996806
 ^FTSE                        54             0.996037,
     symbol              feature_a                feature_b  spearman_rho  \
 54    NG.L        simple_return_1             log_return_1      1.000000   
 0    ^FTSE        simple_return_1             log_return_1      1.000000   
 158  SVT.L        simple_return_1             log_return_1      1.000000   
 106  SSE.L        simple_return_1             log_return_1      1.000000   
 1    ^FTSE                  sma_5                    ema_5      0.999943   
 159  SVT.L                  sma_5                    ema_5      0.999936   
 51   ^FTSE  return_volatility_100  realised_volatility_100      0.999918   
 55    NG.L                  sma_5                    ema_5

In [5]:
# Inspect a small, labelled sample; this is descriptive output, not a predictive assessment.
example_symbol = next(iter(financial_features))
financial_features[example_symbol].tail(5)

,simple_return_1,log_return_1,momentum_5,sma_5,ema_5,price_to_sma_5,return_volatility_5,realised_volatility_5,downside_volatility_5,drawdown_5,...,return_volatility_100,realised_volatility_100,downside_volatility_100,drawdown_100,volume_change_1,relative_volume_5,relative_volume_10,relative_volume_20,relative_volume_50,relative_volume_100
observation_date,,,,,,,,,,,,,,,,,,,,,
2026-10-27,-0.007504,-0.007532,-0.004332,10724.420117,10721.977778,-0.003536,0.005590,0.011346,0.003833,-0.007504,...,0.006281,0.062591,0.004368,-0.020342,-0.058699,0.711059,0.713216,0.709599,0.700186,0.667041
2026-10-28,0.000749,0.000748,-0.005431,10712.740039,10712.818519,-0.001703,0.005476,0.011219,0.003833,-0.006761,...,0.006111,0.060972,0.004135,-0.019609,0.309615,1.018190,0.952094,0.942662,0.918680,0.876006
2026-10-29,0.002581,0.002577,-0.000270,10712.159961,10715.912216,0.000928,0.005605,0.011214,0.003653,-0.004198,...,0.006110,0.060968,0.004135,-0.017079,0.208286,1.242572,1.137440,1.160295,1.112344,1.060591
2026-10-30,0.000718,0.000718,0.003685,10720.039844,10720.541412,0.000910,0.005315,0.010762,0.003356,-0.003483,...,0.006095,0.060784,0.004135,-0.016373,0.032794,1.303329,1.201976,1.205400,1.144678,1.094100
2026-11-02,NaN,NaN,NaN,NaN,10720.541412,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,-1.000000,0.000000,0.000000,0.000000,0.000000,0.000000


## 9. Validation and interpretation

The associated unit tests validate arithmetic and temporal behaviour on small deterministic inputs. The checks below are notebook-level integrity checks; they do not establish historical price correctness, forecasting value, or successful execution until this notebook is actually run.

In [6]:
for symbol, features in financial_features.items():
    assert features.index.equals(market_data[symbol].index)
    assert features.columns.is_unique
    assert not any("target" in name.lower() for name in features.columns)

    # A close-timestamped one-session return at t must equal Close[t] / Close[t-1] - 1.
    expected = market_data[symbol]["Close"].pct_change(fill_method=None)
    pd.testing.assert_series_equal(
        features["simple_return_1"], expected.rename("simple_return_1"),
        check_names=True,
    )

print("Feature index, uniqueness and one-session return checks passed for loaded inputs.")

Feature index, uniqueness and one-session return checks passed for loaded inputs.


## 10. Findings, limitations and unresolved work

This implementation establishes deterministic calculations for close-to-close price features and volume transformations. It does not establish that any feature adds predictive value. That requires a later chronological evaluation and is outside this notebook’s scope.

Limitations carried forward:

- Close-to-close returns exclude cash dividends; corporate-action and source-revision effects remain relevant.
- Historical denomination validation for NG.L, SSE.L and SVT.L remains pending. Scale-free features avoid a currency conversion, but price-level fields must not be interpreted as GBP.
- Missing sessions remain missing. A rolling window counts observed rows, not a separately validated exchange-calendar span.
- High–low range and true range are deferred pending a consistent OHLC adjustment convention.
- No external variables, regimes, targets, model fitting or strategy logic are included.
- The executed descriptive Spearman screen found many highly correlated pairs; see Section 9. It supports treating these outputs as a candidate library and reducing redundant variants before any later chronological model-selection work. Predictive incremental value remains untested.

### Evidence status

- **Implemented in this change:** feature formulas in the Python module and deterministic unit tests.
- **Executed on 14 November 2026:** data loading, feature generation, the redundancy screen and integrity checks; outputs are stored here. The 12 targeted feature tests passed after the zero-volume denominator edge case was handled.
- **Reused historical evidence:** Phase 3 quality and temporal assessments, Phase 5 Notebook 09 analyses, and D140 in the decision log.
- **Not validated:** predictive usefulness, historical revision stability, and corporate-action completeness.
